[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adan-rs/amd/blob/main/notebooks/15_ANOVA.ipynb)

# ANOVA de un factor

*¿Para qué se utiliza?*
El Análisis de Varianza (ANOVA) es una técnica estadística utilizada para comparar las medias de dos o más grupos y determinar si existen diferencias significativas entre ellas. Aunque el t de Student puede compararse con ANOVA cuando solo hay dos grupos, el ANOVA es especialmente útil cuando se comparan tres o más.

*¿Cómo funciona?*
El ANOVA evalúa la proporción de la variabilidad total de los datos que se debe a diferencias entre los grupos (causadas por la variable independiente) y la que se debe a diferencias dentro de los grupos (variabilidad aleatoria). Esta relación se resume en el estadístico F. Un valor F grande sugiere que hay más variabilidad explicada por el grupo al que pertenece cada observación que la atribuible al azar.

*Tipos de ANOVA:*
- ANOVA de un factor (one-way ANOVA): Compara la media de una variable cuantitativa entre niveles de un solo factor categórico (por ejemplo, ingreso promedio por nivel educativo).
- ANOVA de dos o más factores (factorial ANOVA): Evalúa simultáneamente el efecto de varios factores independientes y sus posibles interacciones.
- ANCOVA (Análisis de Covarianza): Extiende el ANOVA al incluir covariables cuantitativas para controlar efectos de otras variables.
- MANOVA (Análisis Multivariado de Varianza): Se utiliza cuando hay dos o más variables dependientes que se analizan de forma conjunta.

*Variables consideradas:*
- Una variable dependiente cuantitativa (de intervalo o razón).
- Una o más variables independientes (factores) categóricas con dos o más niveles (grupos o condiciones).

*Hipótesis planteadas (para ANOVA de un factor):*
- Hipótesis nula (H₀): Todas las medias poblacionales son iguales.
- Hipótesis alternativa (H₁): Al menos una de las medias es diferente.

*Supuestos o requisitos principales:*
- Independencia: Las observaciones dentro y entre los grupos deben ser independientes.
- Normalidad: La variable dependiente debe estar distribuida normalmente dentro de cada grupo.
- Homogeneidad de varianzas: Las varianzas de la variable dependiente deben ser aproximadamente iguales en todos los grupos.
Nota: El ANOVA es robusto ante desviaciones moderadas de los supuestos de normalidad y homogeneidad de varianzas, especialmente cuando los tamaños muestrales son similares entre grupos.

*Criterio de decisión:*
Si el valor p asociado al estadístico F es menor que el nivel de significancia (por ejemplo, α = 0.05), se rechaza la hipótesis nula y se concluye que al menos una media difiere. Sin embargo, el ANOVA no indica cuál o cuáles grupos difieren. Para eso, deben aplicarse pruebas post-hoc (como Tukey, Bonferroni o Scheffé) o realizar comparaciones específicas planificadas.


## Caso de uso
Una cadena de comida rápida planea agregar un nuevo producto a su menú, pero están indecisos entre tres posibles campañas de mercadotecnia. Como un experimento, el producto fue introducido en varias ubicaciones seleccionadas aleatoriamente utilizando diferentes campañas. Se registraron las ventas del nuevo producto por las primeras cuatro semanas. El archivo "marketing.csv" contiene las siguientes variables:  
- MarketID: identificador del mercado
- MarketSize: tamaño del mercado de acuerdo a las ventas
- LocationID: identificador de la ubicación de la tienda
- AgeOfStore: antigüedad de la tienda en años
- Promotion: promoción (1 de 3) que fue probada
- week: semana en que se llevó a cabo la promoción.
- SalesInThousands: ventas para una ubicación específica (LocationID), promoción (Promotion) y semana (week).

Para comparar las ventas de acuerdo a las diferentes promociones, utilizaremos una ANOVA  
Hipótesis nula: todas las medias son iguales.  
Hipótesis alternativa: al menos un par es diferente.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('https://github.com/adan-rs/amd/raw/main/data/marketing.csv')

### Exploración inicial

In [ ]:
df.info()

In [ ]:
df['Promotion'].value_counts()

In [ ]:
df.sample(4)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(x="Promotion", y="SalesInThousands", data=df)
plt.title("Ventas por tipo de promoción")
plt.show()

## Supuestos
### Supuesto de normalidad
Evaluamos si los datos en cada grupo siguen una distribución aproximadamente normal

In [ ]:
from scipy.stats import shapiro

def normalidad_por_grupo(df, group_column, value_column, alpha=0.05):
    """
    Prueba de normalidad (Shapiro-Wilk).
    H0: Los datos provienen de una distribución normal.
    H1: Los datos no siguen una distribución normal.
    """

    grupos = df[group_column].unique()
    resultados = {}

    for g in grupos:
        datos = df[df[group_column] == g][value_column]
        estadistico, p = shapiro(datos)
        
        print(f"\nGrupo: {g}")
        print(f"W = {estadistico:.4f}")
        print(f"p-value = {p:.4f}")
        
        if p < alpha:
            print("Se rechaza H0 → No sigue distribución normal.")
        else:
            print("No se rechaza H0 → Compatible con normalidad.")

In [ ]:
normalidad_por_grupo(df,'Promotion', 'SalesInThousands')

Sin embargo, varios estudios muestran que la prueba ANOVA es robusta ante violaciones de la normalidad si:
- Los grupos tienen un tamaño similar
- Hay por lo menos 40 observaciones en cada grupo.

### Supuesto de igualdad de varianzas

In [ ]:
from scipy.stats import levene

def homogeneidad_varianzas(df, group_column, value_column, alpha=0.05):
    """
    Prueba de Levene (igualdad de varianzas).

    H0: Las varianzas poblacionales son iguales.
    H1: Al menos una varianza es diferente.
    """

    grupos = df[group_column].unique()
    valores = [df[df[group_column] == g][value_column] for g in grupos]

    estadistico, p = levene(*valores)

    print("Prueba de homogeneidad de varianzas (Levene)\n")
    print(f"Estadístico = {estadistico:.4f}")
    print(f"p-value = {p:.4f}")

    if p < alpha:
        print("Se rechaza H0 → Las varianzas NO son iguales.")
    else:
        print("No se rechaza H0 → Varianzas comparables.")

In [ ]:
homogeneidad_varianzas(df,'Promotion', 'SalesInThousands')

## ANOVA

In [ ]:
from scipy.stats import f_oneway

def realizar_anova(df, group_column, measure_column, alpha=0.05):
    """
    ANOVA de una vía.

    H0: Todas las medias poblacionales son iguales.
    H1: Al menos una media es diferente.
    """
    
    grupos = df[group_column].unique()
    valores = [df[df[group_column] == g][measure_column] for g in grupos]

    estadistico, p = f_oneway(*valores)

    print(f"F = {estadistico:.4f}")
    print(f"p-value = {p:.4f}")

    if p < alpha:
        print("Se rechaza H0 → Hay diferencias entre medias.")
    else:
        print("No se rechaza H0 → No hay evidencia de diferencias.")

    return p

In [ ]:
realizar_anova(df,'Promotion', 'SalesInThousands')

In [ ]:
df.groupby("Promotion")["SalesInThousands"].mean()

Si las varianzas no son iguales, y los grupos son muy desbalanceados, una opción es la Welch ANOVA

In [ ]:
import statsmodels.stats.oneway as oneway

def welch_anova(df, group_column, value_column, alpha=0.05):
    """
    Welch ANOVA (varianzas desiguales).

    H0: Todas las medias poblacionales son iguales.
    H1: Al menos una media es diferente.
    """

    res = oneway.anova_oneway(
        df[value_column],
        df[group_column],
        use_var="unequal")

    F = res.statistic
    p = res.pvalue
    df1 = res.df_num
    df2 = res.df_denom

    print("Welch ANOVA\n")
    print(f"F = {F:.4f}")
    print(f"gl1 = {df1:.2f}, gl2 = {df2:.2f}")
    print(f"p-value = {p:.4f}")

    if p < alpha:
        print("Se rechaza H0. Hay diferencias entre medias.")
    else:
        print("No se rechaza H0. No hay evidencia de diferencias.")

In [ ]:
welch_anova(df,'Promotion', 'SalesInThousands')

## Pruebas post-hoc (solo si ANOVA es significativo)
La tabla ANOVA no indica qué grupo es diferente al resto, sin embargo, las pruebas post-hoc son útiles para detectar qué grupo es diferente al resto. 
- LSD/DMS (Diferencia menos significativa): Es el equivalente a múltiples pruebas t, no se hacen correcciones y los resultados no son precisos.
- Bonferroni: Corrige el nivel de significancia dividiéndolo entre el número de grupos. Es preferible cuando son pocas comparaciones. 
- Tukey: Preferible cuando son muchas comparaciones. Es deseable que el tamaño de cada grupo sea igual. 
- REGWQ (Ryan-Einot-Gabriel-Welsh): Recomendable, pero se debe evitar cuando las muestras son de diferente tamaño.
- Dunnett: Es apropiada cuando se desea comparar con un grupo de control.
- Gabriel: Apropiada cuando el tamaño de las muestras es ligeramente diferente.
- GT2 de Hochberg: Apropiadas cuando el tamaño de las muestras es muy diferente
- Games- Howell: Recomendable cuando las varianzas son diferentes.

La librería statsmodel permite realizar la prueba de Tukey, Bonferroni y Dunnet

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy import stats
import pandas as pd

def post_hoc(df, group_column, value_column, metodo="tukey", alpha=0.05):
    """
    Comparaciones múltiples (post hoc).

    Se usa después de un ANOVA significativo.
    Compara pares de medias con corrección por múltiples pruebas.
    """

    if metodo == "tukey":
        resultado = pairwise_tukeyhsd(
            endog=df[value_column],
            groups=df[group_column],
            alpha=alpha
        )
        print(resultado)

    elif metodo == "bonferroni":
        grupos = df[group_column].unique()
        for i in range(len(grupos)):
            for j in range(i+1, len(grupos)):
                g1 = df[df[group_column]==grupos[i]][value_column]
                g2 = df[df[group_column]==grupos[j]][value_column]
                stat, p = stats.ttest_ind(g1, g2)
                p_corr = min(p * len(grupos), 1)  # corrección simple
                print(f"{grupos[i]} vs {grupos[j]} | p ajustado = {p_corr:.4f}")

    else:
        raise ValueError("Método no válido: usar 'tukey' o 'bonferroni'")

In [ ]:
post_hoc(df, "Promotion", "SalesInThousands", metodo='tukey')


In [ ]:
post_hoc(df, "Promotion", "SalesInThousands", metodo='bonferroni')

Ejemplo de redacción de conclusiones: 

>Se realizó un ANOVA de un factor para analizar cómo ________ influye en _________. Los resultados muestran que _________tiene un efecto significativo en __________, F(_,_) = ___, p = ___. Las comparaciones post-hoc utilizando el método de Tukey HSD indican que la media de ______ es significativamente diferente a la media de _________ y _______

## Prueba Kruskal-Wallis (alternativa no paramétrica)
*¿Para qué se utiliza?*
La prueba de Kruskal-Wallis es una alternativa no paramétrica al análisis de varianza (ANOVA) de un factor. Se utiliza para comparar si existen diferencias significativas entre tres o más grupos independientes, sin asumir que los datos siguen una distribución normal. Es útil cuando los supuestos del ANOVA no se cumplen, especialmente en presencia de datos asimétricos, ordinales o con valores atípicos.

*¿Cómo funciona?*
En lugar de comparar medias, como lo hace el ANOVA, esta prueba compara las posiciones (rangos) que ocupan los datos en el conjunto total. Los valores de todas las observaciones se ordenan de menor a mayor y se les asignan rangos. Luego, se evalúa si los rangos promedio difieren entre los grupos.

*Variables consideradas:*
- Una variable dependiente ordinal o cuantitativa (de intervalo o razón, pero no necesariamente normal).
- Una variable independiente categórica con tres o más grupos independientes.

*Hipótesis planteadas:*
- Hipótesis nula (H₀): Las distribuciones (o medianas) de los grupos son iguales.
- Hipótesis alternativa (H₁): Al menos uno de los grupos tiene una distribución (o mediana) diferente.

*Supuestos o requisitos principales:*
- Independencia: Las observaciones deben ser independientes dentro y entre los grupos.
- Escala adecuada: La variable dependiente debe ser al menos ordinal (o cuantitativa continua sin normalidad).
- Distribuciones similares: Se asume que las distribuciones de los grupos tienen forma similar si se desea interpretar la prueba como una comparación de medianas.

*Criterio de decisión:*
Se calcula un estadístico H, que se aproxima a una distribución chi-cuadrada con k−1 grados de libertad, donde k es el número de grupos.
- Si el valor p es menor que el nivel de significancia (por ejemplo, α = 0.05), se rechaza la hipótesis nula y se concluye que al menos un grupo difiere de los demás.
- Si el valor p es mayor que 0.05, no se rechaza la hipótesis nula.
Nota: La prueba indica si hay diferencias globales, pero no especifica cuáles grupos difieren. Para ello, se deben aplicar pruebas post-hoc no paramétricas, como las comparaciones múltiples de Dunn con ajuste por Bonferroni.


In [ ]:
from scipy.stats import kruskal

def kruskal_test(df, group_column, value_column, alpha=0.05):
    """
    Kruskal-Wallis (alternativa no paramétrica al ANOVA).

    H0: Las distribuciones (medianas) de los grupos son iguales.
    H1: Al menos un grupo es diferente.
    """

    grupos = df[group_column].unique()
    valores = [df[df[group_column] == g][value_column] for g in grupos]

    estadistico, p = kruskal(*valores)

    print(f"H = {estadistico:.4f}")
    print(f"p-value = {p:.4f}")

    if p < alpha:
        print("Se rechaza H0 → Hay diferencias entre grupos.")
    else:
        print("No se rechaza H0 → No hay evidencia de diferencias.")


Ejemplo de un reporte de resultados
>Se realizó una prueba de Kruskal-Wallis con el objetivo de comparar la rentabilidad mensual (en porcentaje) de tres tipos de portafolios de inversión: conservador, moderado y agresivo.
La muestra consistió en 20 mediciones mensuales por cada tipo de portafolio. Dado que los datos no seguían una distribución normal (según la prueba de Shapiro-Wilk), se optó por esta prueba no paramétrica.
El resultado fue H(2) = 7.62, con un valor p = 0.0221, lo que indica que existe una diferencia estadísticamente significativa en la mediana de rentabilidad entre al menos dos de los portafolios.


In [ ]:
# Aplicar prueba Kruskal-Wallis
kruskal_test(df, 'Promotion', 'SalesInThousands')

## Integración metodológica

In [ ]:
from scipy.stats import shapiro, levene, f_oneway, kruskal
import numpy as np

def marco_comparacion_grupos(df, group_column, value_column, alpha=0.05):
    """
    Marco de decisión para comparación de grupos.
    Integra supuestos y sugiere prueba adecuada.
    """

    grupos = df[group_column].unique()
    valores = [df[df[group_column] == g][value_column] for g in grupos]
    tamaños = [len(v) for v in valores]

    print("Evaluación de supuestos\n")

    # --- Normalidad ---
    normalidad_global = True
    for g in grupos:
        datos = df[df[group_column] == g][value_column]
        _, p_norm = shapiro(datos)
        if p_norm < alpha:
            normalidad_global = False

    print(f"Normalidad razonable: {'Sí' if normalidad_global else 'Dudosa'}")

    # --- Homogeneidad ---
    _, p_lev = levene(*valores)
    homogeneidad = p_lev >= alpha
    print(f"Varianzas homogéneas: {'Sí' if homogeneidad else 'No'}")

    # --- Balance ---
    balance = max(tamaños) / min(tamaños) < 1.5
    print(f"Muestras balanceadas: {'Sí' if balance else 'No'}\n")

    # --- Recomendación ---
    print("Recomendación metodológica:")

    if homogeneidad and (normalidad_global or balance):
        print("ANOVA es apropiada y robusta en este escenario.")
        estadistico, p = f_oneway(*valores)
        print(f"\nF = {estadistico:.4f}")

    elif not homogeneidad:
        print("Considerar Welch ANOVA (varianzas desiguales).")
        estadistico, p = f_oneway(*valores)
        print(f"\nF (clásico mostrado para referencia) = {estadistico:.4f}")

    else:
        print("Considerar Kruskal-Wallis (posible no normalidad severa).")
        estadistico, p = kruskal(*valores)
        print(f"\nH = {estadistico:.4f}")

    print(f"p-value = {p:.4f}")

    if p < alpha:
        print("Se rechaza H0. Hay diferencias entre grupos.")
    else:
        print("No se rechaza H0. No hay evidencia de diferencias.")

In [ ]:
marco_comparacion_grupos(df, "Promotion", "SalesInThousands")